In [ ]:
import os, warnings, torch

import numpy as np
import scanpy as sc
import pandas as pd

from datasets.data_manager_STARmap_PLUS import AD_Mouse
from utils.notebook_graph_utils import (
    build_graph_model,
    cell_type_gradients,
    feature_gradients,
    load_graph_checkpoint,
    predict_single_graph,
)
from utils.utils import *
from utils.utils_dataloader import *
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


In [ ]:
%run ./args/args_STARmap_PLUS.py
args = args

### Load dataset

In [ ]:
# create the dataset
dataset = AD_Mouse(
    AD_adata_path=args.AD_adata_path,
    Wild_type_adata_path=args.Wild_type_adata_path,
    n_top_genes=args.n_top_genes,
    graph_k=args.graph_k,
    val_ratio=args.val_ratio,
    mask_seed=args.seed,
)


### Load model

In [ ]:
# create the model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_graph_model(args, dataset, use_cell_type=args.use_cell_type).to(device)
model = load_graph_checkpoint(model, 'WholeSliceGraphTransformer_ct_STARmap_PLUS.pth', device=device)


### Inference 

In [ ]:
graph = dataset.testing[0]
pred_value, _, test_mask = predict_single_graph(model, graph, device=device, apply_sigmoid=True)
selected_mask = test_mask & (pred_value[:, 0] > 0.05)
print(f"Selected {int(selected_mask.sum())} nodes for cell-type attribution.")


### Calculate and Visualize the gradients

In [ ]:
embedding_gradients, outputs, attribution_mask = cell_type_gradients(
    model,
    graph,
    target_index=0,
    split='test',
    device=device,
    apply_sigmoid=True,
    selection_mask=selected_mask,
)

grad_norm = embedding_gradients.abs().sum(dim=1).numpy()


In [ ]:
cell_types = dataset.cell_mask

plt.figure(figsize=(7, 6))

data = pd.DataFrame({
    'Category': cell_types.tolist(),
    'Value': grad_norm
})

max_index = np.argmax(grad_norm)

for i in range(len(data)):
    plt.plot([data['Category'][i], data['Category'][i]], [0, data['Value'][i]], color='#004c4c', linewidth=3)

for i in range(len(data)):
    if i == max_index:
        plt.scatter(data['Category'][i], data['Value'][i], color='#004c4c', s=150, marker='^')
    else:
        plt.scatter(data['Category'][i], data['Value'][i], edgecolors='#004c4c', facecolors='none', s=150, marker='^')

plt.title('Cell type analysis for Aβ')
plt.grid(False)
plt.xticks(rotation=45)

plt.savefig('abeta_cell.eps', format='eps', bbox_inches='tight', dpi=300)  
plt.savefig('abeta_cell.png', format='png', bbox_inches='tight', dpi=300)